# Notebook 04 — Desenho do Experimento A/B

**Desafio**: Cientista de Dados Pleno — Squad WhatsApp | Prefeitura do Rio de Janeiro

## Objetivo

**Parte 3** — Propor um experimento controlado que valide se o novo algoritmo de ranking é realmente melhor do que a estratégia atual (envio baseado em ordem de cadastro ou seleção arbitrária).

---

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import norm
import gcsfs

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

BUCKET = 'gs://case_vagas/whatsapp'
print('Ambiente configurado.')

In [ ]:
# Carregar dados para estimar parâmetros históricos
fs = gcsfs.GCSFileSystem(token='anon')

with fs.open(f"{BUCKET.replace('gs://', '')}/base_disparo_mascarado") as f:
    df_disparo = pd.read_parquet(f)

df_disparo['sucesso'] = df_disparo['status_disparo'].isin(['DELIVERED', 'READ']).astype(int)
df_disparo['criacao_envio_datahora'] = pd.to_datetime(df_disparo['criacao_envio_datahora'])

p_baseline = df_disparo['sucesso'].mean()
n_total_hist = len(df_disparo)

print(f'Taxa de entrega histórica (baseline): {p_baseline*100:.2f}%')
print(f'Total de disparos históricos:         {n_total_hist:,}')

---
# 1. Definição do Experimento

## 1.1 Problema de Negócio

**Situação atual**: O motor de disparos seleciona telefones sem otimização — ordem de cadastro, lista completa de números ou critério não documentado.

**Solução proposta**: Novo algoritmo de score que prioriza telefones com base em confiabilidade do sistema de origem, recência do dado, tipo e qualidade.

**Questão**: O novo algoritmo aumenta de fato a taxa de entrega, ou os ganhos observados no backtest são artefatos do dado histórico?

## 1.2 Hipóteses

$$H_0: p_{\text{tratamento}} \leq p_{\text{controle}}$$

$$H_1: p_{\text{tratamento}} > p_{\text{controle}} + \delta$$

onde:
- $p_{\text{controle}}$ = taxa de entrega do sistema atual (baseline histórico)
- $p_{\text{tratamento}}$ = taxa de entrega com o novo algoritmo de seleção
- $\delta$ = efeito mínimo detectável (MDE) — menor melhoria que consideramos economicamente relevante

**Teste**: Teste z de uma cauda (superior) para duas proporções independentes.

In [ ]:
# Parâmetros do experimento
alpha = 0.05      # taxa de erro tipo I (5% de falso positivo)
power = 0.80      # poder estatístico (80% de detectar o efeito real)
delta = 0.02      # MDE: 2 pp de melhoria absoluta sobre o baseline

p_ctrl = p_baseline
p_trat = p_baseline + delta

print(f'Parâmetros do experimento:')
print(f'  α (erro tipo I):          {alpha}')
print(f'  Poder (1-β):              {power}')
print(f'  Baseline (p_controle):    {p_ctrl*100:.2f}%')
print(f'  MDE (δ):                  {delta*100:.1f} pp')
print(f'  Taxa alvo (p_tratamento): {p_trat*100:.2f}%')

---
# 2. Cálculo do Tamanho de Amostra

## 2.1 Fórmula Analítica

Para um teste z de uma cauda com duas proporções:

$$n = \frac{\left(z_{\alpha} \sqrt{2\bar{p}(1-\bar{p})} + z_{\beta}\sqrt{p_c(1-p_c) + p_t(1-p_t)}\right)^2}{(p_t - p_c)^2}$$

onde $\bar{p} = (p_c + p_t)/2$, $z_{\alpha}$ = quantil para $\alpha$ e $z_{\beta}$ = quantil para o poder desejado.

Este $n$ é **por grupo** — o total é $2n$.

In [ ]:
def calcular_tamanho_amostra(p_ctrl, p_trat, alpha, power, two_tailed=False):
    """
    Calcula o tamanho de amostra necessário por grupo para um teste de proporções.
    Retorna n por grupo.
    """
    z_alpha = norm.ppf(1 - alpha) if not two_tailed else norm.ppf(1 - alpha/2)
    z_beta  = norm.ppf(power)

    p_bar = (p_ctrl + p_trat) / 2
    delta_p = p_trat - p_ctrl

    numerador = (z_alpha * np.sqrt(2 * p_bar * (1 - p_bar))
                 + z_beta * np.sqrt(p_ctrl * (1 - p_ctrl) + p_trat * (1 - p_trat)))**2
    n_por_grupo = numerador / delta_p**2

    return int(np.ceil(n_por_grupo))


n_por_grupo = calcular_tamanho_amostra(p_ctrl, p_trat, alpha, power)
n_total = n_por_grupo * 2

print(f'Tamanho de amostra calculado:')
print(f'  Por grupo:  {n_por_grupo:,} CPFs')
print(f'  Total:      {n_total:,} CPFs')

In [ ]:
# Volume diário estimado
volume_diario = df_disparo.groupby(df_disparo['criacao_envio_datahora'].dt.date).size()
media_diaria = volume_diario.mean()

# CPFs únicos por dia (cada CPF pode ter múltiplos disparos)
cpfs_por_dia = df_disparo.groupby(df_disparo['criacao_envio_datahora'].dt.date)['cpf'].nunique()
media_cpf_dia = cpfs_por_dia.mean()

# Fração para o experimento (50/50 split)
cpfs_por_dia_por_grupo = media_cpf_dia / 2

duracao_dias = np.ceil(n_por_grupo / cpfs_por_dia_por_grupo)
duracao_semanas = duracao_dias / 7

print(f'Estimativas de volume:')
print(f'  Média de disparos/dia:      {media_diaria:,.0f}')
print(f'  Média de CPFs únicos/dia:   {media_cpf_dia:,.0f}')
print(f'  CPFs por grupo por dia:     {cpfs_por_dia_por_grupo:,.0f}')
print()
print(f'Duração estimada do experimento:')
print(f'  {duracao_dias:.0f} dias  (~{duracao_semanas:.1f} semanas)')

## 2.2 Curva de Poder × Tamanho de Amostra

Visualizamos como o poder estatístico varia com o tamanho de amostra para diferentes valores de MDE.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Curva 1: Poder × n para diferentes MDEs
n_range = np.logspace(3, 6, 100).astype(int)
mdes = [0.01, 0.02, 0.03, 0.05]
cores_mde = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']

def calcular_poder(n, p_ctrl, delta_p, alpha):
    p_trat = p_ctrl + delta_p
    z_alpha = norm.ppf(1 - alpha)
    p_bar = (p_ctrl + p_trat) / 2
    se_null = np.sqrt(2 * p_bar * (1 - p_bar) / n)
    se_alt = np.sqrt((p_ctrl*(1-p_ctrl) + p_trat*(1-p_trat)) / n)
    z_obs = (delta_p - z_alpha * se_null) / se_alt
    return norm.cdf(z_obs)

for mde, cor in zip(mdes, cores_mde):
    poder = [calcular_poder(n, p_ctrl, mde, alpha) for n in n_range]
    axes[0].plot(n_range, poder, color=cor, linewidth=2, label=f'MDE = {mde*100:.0f} pp')

axes[0].axhline(0.80, color='gray', linestyle='--', alpha=0.7, label='Poder mínimo (80%)')
axes[0].axvline(n_por_grupo, color='#9b59b6', linestyle=':', alpha=0.8,
                label=f'n escolhido: {n_por_grupo:,}')
axes[0].set_xscale('log')
axes[0].set_xlabel('Tamanho de Amostra por Grupo (log)')
axes[0].set_ylabel('Poder Estatístico (1 - β)')
axes[0].set_title('Curva de Poder × Tamanho de Amostra\npor Efeito Mínimo Detectável', fontweight='bold')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[0].legend(fontsize=9)

# Curva 2: Duração do experimento × MDE
mde_range = np.arange(0.005, 0.10, 0.001)
n_necessario = [calcular_tamanho_amostra(p_ctrl, p_ctrl + d, alpha, power) for d in mde_range]
duracao_range = [n / cpfs_por_dia_por_grupo for n in n_necessario]

axes[1].plot(mde_range * 100, duracao_range, color='#3498db', linewidth=2.5)
axes[1].axvline(delta * 100, color='#e74c3c', linestyle='--', alpha=0.8,
                label=f'MDE escolhido: {delta*100:.0f}pp → {duracao_dias:.0f}d')
axes[1].axhline(duracao_dias, color='#e74c3c', linestyle=':', alpha=0.5)
axes[1].axhline(14, color='#f39c12', linestyle='--', alpha=0.6, label='Mínimo razoável: 2 semanas')
axes[1].axhline(56, color='#2ecc71', linestyle='--', alpha=0.6, label='Máximo razoável: 8 semanas')
axes[1].set_xlabel('Efeito Mínimo Detectável (pp)')
axes[1].set_ylabel('Duração estimada (dias)')
axes[1].set_title('Duração do Experimento × MDE\n(Quanto menor o MDE, mais tempo necessário)', fontweight='bold')
axes[1].set_ylim(0, min(duracao_range[0] * 1.1, 500))
axes[1].legend(fontsize=9)

plt.suptitle('Análise de Poder Estatístico do Experimento A/B', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 3. Design Detalhado do Experimento

## 3.1 Estrutura do Teste

In [ ]:
print('=' * 65)
print('        DOCUMENTO DE DESIGN — EXPERIMENTO A/B')
print('        Validação do Algoritmo de Priorização de Telefones')
print('=' * 65)
print()
print('HIPÓTESES')
print(f'  H₀: p_tratamento ≤ p_controle')
print(f'  H₁: p_tratamento > p_controle + {delta*100:.0f} pp  (melhoria mínima relevante)')
print()
print('PARÂMETROS ESTATÍSTICOS')
print(f'  Nível de significância (α):    {alpha}  (5% de falsos positivos tolerados)')
print(f'  Poder (1-β):                   {power}  (80% de detectar efeito real)')
print(f'  MDE:                           {delta*100:.0f} pp absolutas sobre o baseline')
print(f'  Baseline esperado:             {p_ctrl*100:.2f}%')
print(f'  Taxa alvo no tratamento:       {p_trat*100:.2f}%')
print()
print('TAMANHO DE AMOSTRA')
print(f'  Por grupo:  {n_por_grupo:,} CPFs')
print(f'  Total:      {n_total:,} CPFs')
print()
print('DURAÇÃO ESTIMADA')
print(f'  {duracao_dias:.0f} dias  (~{duracao_semanas:.1f} semanas)')
print()
print('UNIDADE DE RANDOMIZAÇÃO')
print('  CPF (não disparo individual)')
print('  → Evita contaminação: o mesmo cidadão sempre recebe o mesmo tratamento')
print()
print('ESTRATIFICAÇÃO')
print('  Por categoria_hsm (tipo de campanha)')
print('  Justificativa: campanhas diferentes têm taxas muito distintas;')
print('  estratificar garante distribuição balanceada entre grupos')
print()
print('GRUPOS')
print('  Controle   (50%): seleção atual — primeiro telefone cadastrado / aleatório')
print('  Tratamento (50%): seleção pelo novo algoritmo de score')
print('=' * 65)

## 3.2 Métricas do Experimento

In [ ]:
print('MÉTRICAS')
print()
print('PRIMÁRIA:')
print('  Taxa de entrega = (DELIVERED + READ) / total_disparos')
print('  Decisão: rejeitar H₀ se z_obs > z_α e p-valor < 0.05')
print()
print('SECUNDÁRIAS:')
print('  1. Taxa de READ:  READ / total_disparos')
print('     (Mede engajamento além da entrega)')
print()
print('  2. Custo por entrega bem-sucedida: custo_total / entregas_bem_sucedidas')
print('     (Impacto financeiro direto)')
print()
print('  3. Tempo até entrega: mediana do tempo entre envio e confirmação de entrega')
print('     (Qualidade da janela de comunicação)')
print()
print('  4. Taxa por categoria_hsm: taxa_entrega por tipo de campanha')
print('     (Verifica se o ganho é uniforme ou concentrado em campanhas específicas)')
print()
print('GUARDRAILS (métricas que não devem piorar):')
print('  - Taxa de FAILED por "User is not on WhatsApp": não deve aumentar > 5%')
print('    relativo — indicaria deterioração do banco de números')
print('  - Taxa de erros técnicos (timeout, network): baseline ± 1%')
print('    (Para distinguir falha do dado vs falha técnica)')

## 3.3 Análise Exploratória para Estratificação

Verificamos as diferenças de taxa por categoria de campanha — justificativa para estratificação.

In [ ]:
# Taxa de entrega por categoria_hsm
if 'categoria_hsm' in df_disparo.columns:
    cat_stats = df_disparo.groupby('categoria_hsm').agg(
        n=('sucesso', 'count'),
        taxa=('sucesso', 'mean')
    ).reset_index().sort_values('taxa', ascending=False)

    cat_stats['pct_volume'] = cat_stats['n'] / cat_stats['n'].sum() * 100

    # Mostrar apenas categorias com volume relevante (> 1% do total)
    cat_relevante = cat_stats[cat_stats['pct_volume'] >= 1].head(15)

    print(f'Variação de taxa de entrega por categoria_hsm:')
    print(f'  Mínima: {cat_relevante["taxa"].min()*100:.1f}%')
    print(f'  Máxima: {cat_relevante["taxa"].max()*100:.1f}%')
    print(f'  Amplitude: {(cat_relevante["taxa"].max() - cat_relevante["taxa"].min())*100:.1f} pp')
    print()
    print('Isso justifica a estratificação por categoria_hsm.')

    fig, ax = plt.subplots(figsize=(14, 5))
    cores = plt.cm.RdYlGn(cat_relevante['taxa'].values / cat_relevante['taxa'].max())
    ax.bar(range(len(cat_relevante)), cat_relevante['taxa'] * 100, color=cores, edgecolor='white')
    ax.axhline(p_ctrl * 100, color='gray', linestyle='--', alpha=0.7, label=f'Média geral: {p_ctrl*100:.1f}%')
    ax.set_xticks(range(len(cat_relevante)))
    ax.set_xticklabels(cat_relevante['categoria_hsm'], rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Taxa de Entrega (%)')
    ax.set_title('Taxa de Entrega por Categoria de Campanha (HSM)\n'
                 'Amplitude alta → Estratificação obrigatória no A/B', fontweight='bold')
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Coluna categoria_hsm não encontrada — verificar nome da coluna nos dados reais.')

---
# 4. Protocolo de Análise e Tomada de Decisão

## 4.1 Teste Estatístico

In [ ]:
def analisar_ab(n_ctrl, k_ctrl, n_trat, k_trat, alpha=0.05):
    """
    Análise do experimento A/B ao final.
    n = tamanho do grupo, k = número de sucessos.
    Retorna: z-score, p-valor, IC do efeito, decisão.
    """
    p_c = k_ctrl / n_ctrl
    p_t = k_trat / n_trat
    delta_obs = p_t - p_c

    # Proporção pooled (sob H₀: p_ctrl = p_trat)
    p_pool = (k_ctrl + k_trat) / (n_ctrl + n_trat)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n_ctrl + 1/n_trat))

    z = delta_obs / se
    p_valor = 1 - norm.cdf(z)  # teste unicaudal

    # IC 95% do efeito observado
    se_ic = np.sqrt(p_c*(1-p_c)/n_ctrl + p_t*(1-p_t)/n_trat)
    ic_lower = delta_obs - 1.96 * se_ic
    ic_upper = delta_obs + 1.96 * se_ic

    decisao = 'REJEITAR H₀ — novo algoritmo é melhor' if p_valor < alpha else 'NÃO REJEITAR H₀'

    return {
        'p_controle': p_c,
        'p_tratamento': p_t,
        'delta_obs': delta_obs,
        'z_score': z,
        'p_valor': p_valor,
        'ic_95_lower': ic_lower,
        'ic_95_upper': ic_upper,
        'decisao': decisao
    }


# Simulação: e se observássemos os resultados esperados?
print('Simulação: resultado esperado se o algoritmo funcionar como previsto')
print()

sim = analisar_ab(
    n_ctrl=n_por_grupo, k_ctrl=int(n_por_grupo * p_ctrl),
    n_trat=n_por_grupo, k_trat=int(n_por_grupo * p_trat)
)

for k, v in sim.items():
    if isinstance(v, float):
        if 'p_valor' in k:
            print(f'  {k:<22}: {v:.6f}')
        elif 'delta' in k or 'ic_' in k:
            print(f'  {k:<22}: {v*100:+.2f} pp')
        elif 'p_' in k:
            print(f'  {k:<22}: {v*100:.2f}%')
        else:
            print(f'  {k:<22}: {v:.3f}')
    else:
        print(f'  {k:<22}: {v}')

## 4.2 Monitoramento Contínuo e Early Stopping

In [ ]:
print('PROTOCOLO DE MONITORAMENTO')
print()
print('Frequência de leitura de resultados:')
print('  - Diária: métricas de guardrail (não deve aumentar FAILED técnico)')
print('  - Semanal: review de métricas primária e secundárias')
print()
print('Early stopping por dano:')
print('  Se em qualquer semana a taxa de FAILED do grupo tratamento superar')
print('  o controle em > 2 pp absolutos com p < 0.01, encerrar o experimento.')
print()
print('Early stopping por sucesso (opcional):')
print('  Utilizar correção de O\'Brien-Fleming para múltiplas leituras:')
print('  α* = α × f(número de leituras) — mantém α total controlado.')
print('  Recomendação: apenas 1 look intermediário na metade do período,')
print('  com limiar de α* = 0.005 (conservador).')
print()
print('Análise final:')
print('  Ao atingir o tamanho de amostra planejado:')
print('  1. Teste z unicaudal (α = 0.05)')
print('  2. IC 95% do efeito observado')
print('  3. Análise por subgrupos (categoria_hsm) para homogeneidade do efeito')
print('  4. Análise de sensibilidade: resultado muda se excluirmos falhas técnicas?')

## 4.3 Visualização do Cronograma

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

semanas = np.arange(0, duracao_semanas + 1, 0.5)

# Tamanho de amostra acumulado por grupo
n_acumulado = np.minimum(semanas * cpfs_por_dia_por_grupo * 7, n_por_grupo)

# Poder acumulado
poder_acumulado = []
for n in n_acumulado:
    if n < 10:
        poder_acumulado.append(alpha)
    else:
        p = calcular_poder(max(int(n), 1), p_ctrl, delta, alpha)
        poder_acumulado.append(p)

ax2 = ax.twinx()

ax.fill_between(semanas, n_acumulado, alpha=0.3, color='#3498db')
ax.plot(semanas, n_acumulado, color='#3498db', linewidth=2, label='CPFs por grupo (acumulado)')
ax.axhline(n_por_grupo, color='#3498db', linestyle='--', alpha=0.6, label=f'Meta: {n_por_grupo:,} CPFs')

ax2.plot(semanas, poder_acumulado, color='#e74c3c', linewidth=2.5, label='Poder estatístico')
ax2.axhline(0.80, color='#e74c3c', linestyle=':', alpha=0.6, label='Poder mínimo (80%)')

# Marco look intermediário
look_semana = duracao_semanas / 2
ax.axvline(look_semana, color='#f39c12', linestyle='--', alpha=0.7, label=f'Look intermediário: semana {look_semana:.0f}')
ax.axvline(duracao_semanas, color='#2ecc71', linestyle='--', alpha=0.7, label=f'Análise final: semana {duracao_semanas:.0f}')

ax.set_xlabel('Semanas de Experimento')
ax.set_ylabel('CPFs por grupo (acumulado)', color='#3498db')
ax2.set_ylabel('Poder estatístico (1-β)', color='#e74c3c')
ax.set_title('Cronograma do Experimento: Acúmulo de Amostra e Poder Estatístico',
             fontweight='bold', fontsize=12)

ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0%}'))

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=9)

plt.tight_layout()
plt.show()

---
# 5. Resumo Executivo do Experimento

In [ ]:
# Estimar impacto financeiro potencial
# Assumindo custo por disparo e valor por entrega bem-sucedida
# (valores fictícios para fins de estimativa)

print('=' * 65)
print('            RESUMO EXECUTIVO DO EXPERIMENTO A/B')
print('=' * 65)
print()
print('OBJETIVO')
print('  Validar que o novo algoritmo de priorização de telefones')
print('  aumenta a taxa de entrega das mensagens WhatsApp da Prefeitura.')
print()
print('DESIGN')
print(f'  Tipo:         Experimento controlado, randomizado')
print(f'  Split:        50% controle / 50% tratamento')
print(f'  Unidade:      CPF (randomização por cidadão)')
print(f'  Estratos:     Categoria de campanha (categoria_hsm)')
print()
print('PARÂMETROS')
print(f'  H₀:           p_trat ≤ p_ctrl (algoritmo não melhora)')
print(f'  H₁:           p_trat > p_ctrl + {delta*100:.0f}pp')
print(f'  α:            {alpha} | Poder: {power}')
print(f'  Baseline:     {p_ctrl*100:.2f}% | Alvo: {p_trat*100:.2f}%')
print()
print('TAMANHO E DURAÇÃO')
print(f'  {n_por_grupo:,} CPFs por grupo  |  {n_total:,} CPFs total')
print(f'  Duração estimada: {duracao_dias:.0f} dias (~{duracao_semanas:.1f} semanas)')
print()
print('MÉTRICAS DE SUCESSO')
print(f'  Primária:  taxa DELIVERED+READ  → p-valor < {alpha}')
print(f'  Sec. 1:    taxa READ')
print(f'  Sec. 2:    custo por entrega bem-sucedida')
print(f'  Sec. 3:    tempo até entrega')
print(f'  Guardrail: taxa FAILED técnica não deve aumentar')
print()
print('IMPACTO POTENCIAL')
ganho_pp = delta * 100
print(f'  Se H₁ confirmada: +{ganho_pp:.1f} pp na taxa de entrega')
print(f'  Sobre volume histórico de {n_total_hist:,} disparos:')
entregas_adicionais = int(n_total_hist * delta)
print(f'  → Aproximadamente {entregas_adicionais:,} entregas adicionais')
print(f'    (cidadãos alcançados que antes não receberiam a mensagem)')
print('=' * 65)

---
# 6. Conclusão

O experimento A/B foi desenhado para:

1. **Isolar o efeito causal** do novo algoritmo — randomização por CPF elimina confundidores
2. **Detectar ganhos relevantes** com poder suficiente (80%) — calculado com base no volume real de disparos
3. **Proteger contra regressões** — guardrails monitoram métricas de qualidade técnica
4. **Permitir tomada de decisão clara** — critério pré-estabelecido antes do experimento (sem data dredging)

**Se o experimento confirmar H₁**: implementar o algoritmo de score como padrão em toda a base de disparos, com revisão semestral dos parâmetros (score de sistema e λ de decaimento) à medida que novos dados são acumulados.

**Se o experimento não rejeitar H₀**: investigar subgrupos (qual categoria_hsm respondeu melhor?) e refinar o modelo antes de nova tentativa.